# Tutorial: demo data → HTML report

Interactive walkthrough for new users. Structure follows [sc-best-practices · perturbation modeling](https://www.sc-best-practices.org/conditions/perturbation-modeling/).

**Offline-first:** Steps 0–4 use **synthetic** DRAGEN-like files (`write-demo`). No network required for the demo path.

Related docs: [tutorial.md](tutorial.md) (CLI twin) · [CLI ↔ YAML](CLI_YAML.md) · [Statistical caveats](STATISTICAL_CAVEATS.md) · [Step dependencies](ANALYSIS_DEPENDENCIES.md)

## Motivation

You have DRAGEN CRISPR-mode secondary outputs (MEX + guide assignments) and want a tertiary pass: QC, Mixscape, E-distance, DE, and a readable report. Demo data is small (~120 cells). Treat results as a **pipeline smoke test**, not biology.


## Prerequisites

From the **repo root**, with the package importable:

```bash
conda env create -f environment.yml
conda activate perturbseq-tertiary
# or: pip install -e ".[de,dev]" -c constraints.txt
jupyter notebook docs/tutorial.ipynb
# or: jupyter lab docs/tutorial.ipynb
```

If Mixscape / E-distance are unavailable (no pertpy), set `SKIP_PERTPY = True` below — the demo still completes QC + exploratory DE.


In [ ]:
from pathlib import Path
import sys

# Resolve repo root whether the kernel cwd is repo root or docs/
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "perturbseq").is_dir() else HERE.parent
SRC = ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DEMO_DIR = ROOT / "data" / "demo_nb"
OUT_DIR = ROOT / "results" / "demo_nb"
SAMPLE_ID = "sample1"

# True = skip Mixscape + E-distance (core-only / no pertpy)
SKIP_PERTPY = False

print("ROOT =", ROOT)
print("DEMO_DIR =", DEMO_DIR)
print("OUT_DIR =", OUT_DIR)


## Step 1 — Write demo inputs

Creates a minimal DRAGEN-shaped bundle (`sample1.scRNA.*`) with NegCtrl + targeting guides, singletons, and a few multiplets / empty assignments.


In [ ]:
from perturbseq.demo import write_demo_dragen

path = write_demo_dragen(DEMO_DIR)
print("Wrote:", path)
print("Files:")
for p in sorted(DEMO_DIR.iterdir()):
    if p.name.startswith("."):
        continue
    print(" ", p.name)


## Step 2 — Run the tertiary pipeline

Uses `PipelineConfig` + `run_pipeline` (same path as `python -m perturbseq run`). Demo defaults: control `NT`, singlet-only, Mixscape on when pertpy is present.


In [ ]:
from perturbseq import PipelineConfig, run_pipeline
from perturbseq._deps import pertpy_available

skip_mix = SKIP_PERTPY or not pertpy_available()
skip_dist = SKIP_PERTPY or not pertpy_available()
if skip_mix and not SKIP_PERTPY:
    print("pertpy not available → skipping Mixscape / E-distance")

config = PipelineConfig(
    input_dir=DEMO_DIR,
    output_dir=OUT_DIR,
    sample_id=SAMPLE_ID,
    control="NT",
    singlet_only=True,
    skip_mixscape=skip_mix,
    skip_distance=skip_dist,
    n_jobs=1,
    random_state=0,
)
report = run_pipeline(config, resume=False)
print("HTML:", OUT_DIR / "report.html")
print("h5ad:", OUT_DIR / f"{SAMPLE_ID}.tertiary.h5ad")
print("n_after_singlet:", report.get("n_after_singlet"))


Equivalent CLI (from repo root):

```bash
python -m perturbseq write-demo --output-dir data/demo_nb
python -m perturbseq run \
  --config configs/default.yaml \
  --input-dir data/demo_nb \
  --output-dir results/demo_nb \
  --sample-id sample1
```

Faster smoke test: add `--skip-mixscape --skip-distance`. Resume: add `--resume`.


## Step 3 — Inspect intermediate outputs


In [ ]:
import json
import pandas as pd
import scanpy as sc

h5ad = OUT_DIR / f"{SAMPLE_ID}.tertiary.h5ad"
adata = sc.read_h5ad(h5ad)
print(adata)
print(adata.obs[["guide_id", "gene_target", "perturbation"]].head())

tables = OUT_DIR / "tables"
comp = tables / "composition_audit.csv"
if comp.exists():
    print(pd.read_csv(comp).to_string(max_rows=20))

manifest = OUT_DIR / "run_manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text())
    print("manifest keys:", sorted(m.keys())[:12])


**Suggested inspection order**

1. `run_manifest.json` / `run.log` — stage success, versions, checksums
2. `tables/composition_audit.csv` — cells remaining after QC / singlet / Mixscape
3. `tables/edistance.csv` (+ `pre_mixscape/` / `post_mixscape/` if Mixscape ran)
4. `tables/etest.csv` — permutation p / padj
5. `tables/de_*.csv` — exploratory Wilcoxon without replicates
6. `tables/guide_qc.csv`, `gene_guide_consistency.csv`, `qc_warnings.csv`
7. `figures/` — QC, UMAP (viz only), volcano, E-distance
8. `*.tertiary.h5ad` — `obs` guide columns; `layers['counts']` raw; `X` log-norm


## Step 4 — Open the report

Open `results/demo_nb/report.html` in a browser (macOS: `open results/demo_nb/report.html`).

Rebuild HTML/JSON without re-running analysis:


In [ ]:
from perturbseq.report import write_analysis_report

summary = write_analysis_report(OUT_DIR, control="NT")
print("perturbations:", summary.get("perturbation_summary", {}).get("n_perturbations"))
print("Wrote:", OUT_DIR / "report.html")


## 常见坑 / Common pitfalls

Full discussion + literature: **[STATISTICAL_CAVEATS.md](STATISTICAL_CAVEATS.md)**.

| Pitfall | Why it happens | What to do |
| --- | --- | --- |
| **Multi-guide inconsistency** | Same gene, different guides → discordant effect directions (assignment quality, UMI, incomplete edit, off-target, cytotoxicity) | Inspect `gene_guide_consistency.csv` / `qc_warnings.csv`; do **not** pool inconsistent guides into one “gene KO” claim |
| **Mixscape labels many NP** | Transcriptomic classifier: targeting barcode but still resembles NT pool; pool-dependent; not proof the guide “failed” | Read composition audit; NP ≠ biological null; see caveats |
| **No FDR / replication claims** | Cells ≠ biological replicates; without `replicate_col` the pipeline emits exploratory Wilcoxon | Use ranks for hypotheses only; need ≥2 true replicates for pseudobulk + PyDESeq2 |
| **UMAP / Leiden as “effect”** | Global structure mixes batch, cell cycle, guide load, ambient RNA | Prefer E-distance / E-test (+ replicate-aware DE) |
| **Filtering changes hits** | MAD QC, singlet filter, Mixscape KO+NT subset drop cells | Compare stages in `composition_audit.csv` |
| **Trusting DRAGEN assignment blindly** | Tertiary pipeline does not re-call guides by default | Assignment quality bounds everything downstream |

Demo NegCtrl / targeting modules are **synthetic** — do not over-interpret gene names.


## Step 5 — Optional follow-ons


In [ ]:
# Optional: cell-cycle / state annotation only
# from perturbseq.cli import main
# main(["annotate", "--h5ad", str(h5ad), "--output-dir", str(OUT_DIR), "--inplace"])

# Optional: guide QC only
# main(["guide-qc", "--h5ad", str(h5ad), "--output-dir", str(OUT_DIR), "--control", "NT"])
print("Uncomment above if you want annotate / guide-qc on the demo h5ad.")


## Optional — public dataset (Papalexi 2021 via pertpy)

`# optional` — **requires network** on first download and `pertpy`. This is **not** DRAGEN MEX input; it shows how to inspect a published ECCITE-seq object for comparison with Mixscape literature. This tertiary pipeline still expects DRAGEN-shaped files for `run`.


In [ ]:
# optional — needs pertpy + network (first download)
# Papalexi et al., Nat Genet 2021 — Mixscape / ECCITE-seq reference dataset

try:
    import pertpy as pt
except ImportError:
    print("pertpy not installed; skip this cell. pip install -e '.[de]' -c constraints.txt")
else:
    mdata = pt.data.papalexi_2021()
    print(mdata)
    # RNA modality is typically under mdata.mod["rna"] (MuData)
    if hasattr(mdata, "mod") and "rna" in mdata.mod:
        rna = mdata.mod["rna"]
        print("RNA:", rna)
        print("obs columns (sample):", list(rna.obs.columns)[:12])
    else:
        print("Inspect mdata.obs / mdata.mod for guide / gene labels.")


When ready for real data, point `--input-dir` at a DRAGEN sample folder and keep the same inspection order. Flag map: [CLI_YAML.md](CLI_YAML.md).
